# Ai-Plan: one noise proposal, one complete future plan

This tour loads **real saved artifacts** from bounded CLI experiments. The decoders are frozen analytic functions, not learned DSBM generators. The proposal and antisymmetric ratio networks are learned. Reopening the notebook never launches training or simulation.

Run short demos first if outputs do not exist:
```bash
uv run ai-planner demo analytic --config configs/analytic_smoke.yaml
uv run ai-planner demo routes --config configs/route_smoke.yaml
```
Three clocks remain distinct: physical execution, future decoder revision time (absent here), and MCMC refinement of a whole candidate chunk.

In [ ]:
from pathlib import Path
import json
import numpy as np
from IPython.display import display, Image, HTML, Markdown
from ai_planner.visualization import render

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
RUNS = sorted(p.parent for p in (ROOT / "outputs").glob("*/results.json"))
if not RUNS:
    raise FileNotFoundError("Run a CLI demo first; this notebook never starts experiments.")
print("Available runs:", *RUNS, sep="\n")
RUN = RUNS[-1]  # Change explicitly to inspect another saved run.
print("Selected:", RUN)
manifest = render(RUN)  # Loads arrays only: zero oracle evaluations.
def figure(name):
    path = RUN / "figures" / f"{name}.png"
    if path.exists():
        display(Image(filename=str(path)))
    else:
        display(Markdown(f"*{name}: not measured in this selected run.*"))

## 1. Prior, decoder, cost, and target

The prior is $z\sim\mathcal N(0,I)$. The map $y=2\Phi(z)-1$ gives a uniform descriptor prior on the square. The noise target is $p(z)\propto\varphi(z)\exp[-J(G(z))/\beta]$. Descriptor density is proportional to the cost weight, **not the cost itself**. No decoder Jacobian is added to the noise-space MH ratio.

Fine-grid quadrature and independent rejection samples provide reference checks. Region masses include the energy floor; they are not the mixture weights.

In [ ]:
figure("target_and_prior")
checks = RUN / "reference_checks.json"
if checks.exists():
    for row in json.loads(checks.read_text()):
        print(row["context"], "grid mass change:", row["region_mass_grid_difference"],
              "max rejection standard errors:", row["rejection_max_standardized_error"])

## 2. Replay without recomputation

Use context, temperature, method, refresh-rate and step controls. Only saved combinations are offered. Exact acceptance values for learned transitions come from separately charged audit evaluations; unavailable values remain blank. A rejection repeats the Markov state. Route frames are restricted to transitions with saved physical arrays.

In [ ]:
replay = RUN / "figures" / "replay.html"
display(HTML(replay.read_text()))
# The standalone HTML also opens directly in a browser, without a CDN or server.

## 3. Acceptance is not exploration

Tiny local proposals can accept nearly every move while remaining inside one mode. Inspect mode switches, ESS and occupancy. Dotted reference masses belong to the matching target. Rejected states remain in traces. Histogram TV is coarse and sample-dependent, not exact continuous TV.

In [ ]:
figure("chain_and_region_occupancy")
figure("sampler_vs_planner_summary")

## 4. Ratio calibration and exact refresh

The ratio is antisymmetric by construction: $d(z,z')=g(z,z')-g(z',z)$. Raw squared error fits exact labels with the proposal frozen; changing the proposal requires refreshed labels. A controlled variant injects an antisymmetric error.

Refresh is a state-independent Bernoulli mixture $(1-\rho)T+\rho P_{ref}$. Larger refresh probability spends oracle work. Short finite chains need not improve monotonically; these development comparisons do not establish a learned-method win.

In [ ]:
figure("ratio_calibration")
figure("refresh_bias_tradeoff")

## 5. Exact finite-state proof audit

The small grid is a **separate discrete model**. Proposal rows are normalized on the grid, then discrete Hastings ratios are recomputed. Exact finite sums test accepted-flow KL, one-step bounds, stationary bias and finite-step bounds.

The minorization uses $\kappa=\rho\exp(-J_{max}/\beta)$. Bounds above one are vacuous; raw values are shown. Monte Carlo balance estimates can be negative. A near-identity example demonstrates that small one-step error alone does not certify stationary accuracy.

In [ ]:
figure("finite_grid_bound_audit")
audit_file = RUN / "finite_grid.json"
if audit_file.exists():
    for context, audit in json.loads(audit_file.read_text()).items():
        for variant, row in audit["variants"].items():
            print(context, variant, "flow identity residual:", row["accepted_flow_identity_error"],
                  "one-step TV:", row["one_step_tv"])

## 6. Entire command chunks and physical outcomes

Route replays show the full two-dimensional coefficient space beside commanded targets and realized physical positions. An MCMC proposal changes the entire future chunk from the same immutable snapshot.

The old plan, retained suffix and completed tail show causal memory. Posterior frames show the **historical prefix that created the snapshot**. Methods prefixed `planner_` show the **actually selected new prefix and its execution**. The next retained suffix belongs to that selected chunk. A new physical observation creates a new target and chain.

In [ ]:
figure("route_repair_replay")
figure("success_vs_simulator_steps")
accounting = RUN / "accounting.json"
if accounting.exists():
    print(json.dumps(json.loads(accounting.read_text()), indent=2))

## 7. Sampling and planning answer different questions

Best feasible incumbents are optimizer outputs, not posterior samples. Independent best-of-N and CEM receive the same oracle budget. Learned-only candidates cannot enter the incumbent pool without evaluation; final validation is counted. Screening and fallback change the execution policy.

Training is intentionally absent from notebook execution. Use the CLI pilot configuration for an explicit bounded training run. Read `docs/RUN_REPORT.md` for measured outcomes, failures and inconclusive comparisons, and `docs/DECISIONS.md` for discretionary choices. No DSBM superiority, real-robot safety, or physical passivity is claimed.